# Guida MLOps: Monitoraggio della Reputazione Online di un'Azienda

Questo documento costituisce la guida ufficiale all'architettura e al funzionamento del progetto. Di seguito vengono spiegate in dettaglio le varie componenti del software, il perché delle scelte progettuali e come testare l'applicazione tramite comandi `curl` da terminale.


## 1. Introduzione ed Obiettivo del Progetto

L'obiettivo di questo progetto è realizzare un sistema completo di **Sentiment Analysis** per monitorare la reputazione online del brand in tempo reale.

Nei sistemi di Machine Learning reali, un modello rilasciato in produzione non rimane valido per sempre. Con il passare del tempo si verifica il fenomeno del **Data Drift** (il linguaggio cambia, nascono nuove tendenze o l'utente si esprime in modo diverso). Per questo motivo, una soluzione professionale richiede una pipeline MLOps robusta che gestisca il ciclo di vita del modello:

1. **Servire predizioni a bassa latenza** per analizzare il sentiment delle recensioni degli utenti.
2. **Loggare e monitorare le prestazioni** in produzione salvando le metriche in un database per essere visualizzate su una dashboard.
3. **Aggiornare il modello (Retraining)** periodicamente con dati freschi e certificati senza interrompere l'operatività degli utenti.
4. **Gestire le versioni dei modelli** in modo centralizzato e sicuro tramite un Model Registry nel cloud.

## 2. Componenti del Progetto ed Architettura

L'architettura del software si basa su una netta separazione dei compiti (Separation of Concerns). I componenti principali sono:

### A. Backend FastAPI (`app/main.py`)
È l'applicazione web che espone le API al pubblico. Fornisce l'endpoint per la predizione del sentiment (`/predict`), per consultare il numero di log (`/metrics`), elencare i modelli locali (`/models_list`), cambiare a caldo il modello attivo (`/change_model`) e scatenare un retraining asincrono locale (`/retrain`).

### B. Modulo di Retraining (`app/training.py`)
Contiene la logica effettiva di addestramento. Si occupa di scaricare il dataset da Kaggle tramite le API, formattare i testi, avviare il processo di fine-tuning di un modello Transformer (RoBERTa) ed effettuare il salvataggio sia in locale che nel cloud di HuggingFace Hub.

### C. Script di Esecuzione Offline (`scripts/esegui_training.py`)
È un semplice entry-point in Python che richiama la funzione di retraining. Esiste per poter essere eseguito in modo indipendente in contesti in cui non vogliamo caricare il server web FastAPI (ad esempio su macchine virtuali di calcolo temporanee o all'interno di pipeline di CI/CD).

### D. GitHub Actions (`.github/workflows/auto_train.yml`)
Un flusso di integrazione e rilascio continuo (CI/CD) che ogni domenica notte (o su richiesta manuale) avvia un container virtuale su GitHub, scarica il codice del progetto, effettua l'addestramento offline usando le risorse CPU/GPU di GitHub e pubblica automaticamente il modello aggiornato su HuggingFace Hub.

### E. Database SQLite & Grafana
Ogni predizione effettuata dalle API viene salvata all'istante all'interno di un database SQLite locale (`sentiment_logs.db`). Grafana è configurato per leggere questo DB ed esporre una dashboard con grafici sull'andamento delle opinioni degli utenti, del volume di traffico e della confidenza media del modello.

## 3. Le Scelte Tecniche e Rationale (Il "Perché")

Durante lo sviluppo sono state prese tre decisioni architetturali cruciali:

### A. Perché il retraining non deve girare sincrono all'interno dell'API?
L'addestramento dei Transformer è un'operazione estremamente pesante che richiede minuti o ore e consuma gran parte delle risorse hardware (CPU/GPU e RAM). Se eseguissimo il training in modo sincrono all'interno dell'API, il server FastAPI rimarrebbe bloccato in attesa della fine del processo, impedendo a tutti gli altri utenti di fare predizioni. 
*Soluzione:* Il retraining è gestito in background in modo asincrono (`BackgroundTasks` di FastAPI) o idealmente demandato a flussi di CI/CD esterni.

### B. Perché evitare l'addestramento sul database SQLite locale?
È forte la tentazione di usare i testi immessi dagli utenti (salvati nel DB di log) per ri-addestrare il modello. Tuttavia, questo genera un **Feedback Loop (o Eco-Chamber)**. Se il modello classifica erroneamente una recensione negativa come positiva e noi ri-addestriamo il modello su quella stessa coppia testo-label errata, il modello si convincerà ancora di più del suo errore, portando a una degradazione catastrofica delle prestazioni nel tempo.
*Soluzione:* Il retraining deve basarsi solo su dati freschi certificati provenienti da Kaggle (Ground Truth reale).

### C. Perché usare HuggingFace Hub?
Salvare i pesi del modello (che occupano centinaia di megabyte) nel repository Git o sul server locale è inefficiente e non scalabile. HuggingFace Hub funge da **Model Registry** sicuro e centralizzato, permettendo al server FastAPI di scaricare la versione corretta a caldo tramite un simple ID di repository.

## 4. Guida Pratica al Test degli Endpoint

Di seguito vengono riportati i comandi `curl` per testare singolarmente ogni endpoint esposto dalle nostre API, completi delle risposte JSON che ti devi aspettare dal server.

### Prerequisiti per il Test
Assicurati che lo stack sia attivo. Puoi avviarlo in locale tramite Docker:
```bash
docker-compose up --build
```
Oppure direttamente con Python ed uvicorn (se sei in Codespaces o sul tuo PC locale):
```bash
pip install -r requirements.txt
uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

### Endpoint 1: Verifica dello Stato del Server (`/`)
Restituisce lo stato del backend e indica quale modello è correntemente attivo per le predizioni.

**Comando da terminale:**
```bash
curl -X GET "http://localhost:8000/"
```

**Risposta JSON attesa:**
```json
{
  "status": "online",
  "model": "cardiffnlp/twitter-roberta-base-sentiment-latest"
}
```

### Endpoint 2: Predizione del Sentiment (`/predict`)
Interroga il modello passando un testo. Il server calcola il sentiment, registra la recensione nel database SQLite (`sentiment_logs.db`) e restituisce la risposta.

**Comando da terminale:**
```bash
curl -X POST "http://localhost:8000/predict" \
     -H "Content-Type: application/json" \
     -d "{\"text\": \"Questo prodotto è eccellente, lo consiglio a tutti!\"}"
```

**Risposta JSON attesa:**
```json
{
  "label": "positive",
  "score": 0.9821
}
```

### Endpoint 3: Conteggio delle Predizioni Loggate (`/metrics`)
Interroga il database per sapere quanti record di predizioni sono stati salvati fino ad ora. Grafana usa queste informazioni per popolare i grafici.

**Comando da terminale:**
```bash
curl -X GET "http://localhost:8000/metrics"
```

**Risposta JSON attesa:**
```json
{
  "total_predictions_logged": 1
}
```

### Endpoint 4: Lista dei Modelli Locali (`/models_list`)
Verifica quali versioni del modello sono state addestrate e salvate localmente all'interno della directory `/opt/versioned_models`.

**Comando da terminale:**
```bash
curl -X GET "http://localhost:8000/models_list"
```

**Risposta JSON attesa (senza modelli addestrati in precedenza):**
```json
{
  "models": []
}
```

**Risposta JSON attesa (con modelli addestrati):**
```json
{
  "models": [
    "model_20260617"
  ]
}
```

### Endpoint 5: Cambio della Versione del Modello (`/change_model`)
Consente agli sviluppatori o agli amministratori del sistema di passare a una versione precedente o successiva del modello a caldo, senza dover riavviare il server. Consente anche di reimpostare la versione base.

**Comando da terminale (per passare a una versione locale):**
```bash
curl -X POST "http://localhost:8000/change_model" \
     -H "Content-Type: application/json" \
     -d "{\"version\": \"model_20260617\"}"
```

**Risposta JSON attesa:**
```json
{
  "message": "Modello cambiato con successo alla versione model_20260617."
}
```

**Comando da terminale (per tornare al modello base):**
```bash
curl -X POST "http://localhost:8000/change_model" \
     -H "Content-Type: application/json" \
     -d "{\"version\": \"base\"}"
```

**Risposta JSON attesa:**
```json
{
  "message": "Modello cambiato con successo alla versione base."
}
```

### Endpoint 6: Avvio del Retraining Asincrono (`/retrain`)
Avvia un processo di retraining in background. Risponde immediatamente con lo stato di caricamento avviato.

**Comando da terminale:**
```bash
curl -X POST "http://localhost:8000/retrain"
```

**Risposta JSON attesa:**
```json
{
  "message": "Processo di retraining avviato in background."
}
```

**Nota Bene:** Durante l'esecuzione del retraining in background, se provi a chiamare `/predict` o `/`, il server ti risponderà temporaneamente con:
```json
{
  "status": "training in progress, please try later."
}
```
Questo impedisce che nuove predizioni interferiscano con la CPU/GPU o la memoria RAM sature a causa dell'addestramento in corso.